# Introduction to Recursive-Set

In theoretical computer science, the concept of a *Set* is fundamental to formal models such as automata, grammars, constraint satisfaction problems (CSPs like Sudoku), and propositional logic. Before we can implement algorithms for these structures, we must establish a computationally sound representation of mathematical sets.

## Mathematical Definition
A set is a collection of distinct objects where the order of elements is irrelevant. Formally, for two sets $A$ and $B$, the following holds true:

$$ A = \{1, 2, 3\} $$
$$ B = \{3, 2, 1\} $$
$$ A = B $$

Furthermore, adding an element that already exists in the set does not alter the set. If we define $C = A \cup \{2\}$, then $C$ evaluates to $\{1, 2, 3\}$.

In the following we will examine how these mathematical properties translate to data structures in TypeScript.

In [1]:
let A = new Set([1, 2, 3]);
A

Set(3) { 1, 2, 3 }


In [2]:
let B = new Set([3, 2, 1]);
B

Set(3) { 3, 2, 1 }


In [3]:
A == B

false


## Native TypeScript Sets

TypeScript (and JavaScript) provides a built-in `Set` data structure. The native `Set` successfully enforces the mathematical property of uniqueness whilst working with `string` and `number`.

In [4]:
const primitiveSet = new Set<number>();

primitiveSet.add(1);
primitiveSet.add(2);
primitiveSet.add(3);
primitiveSet.add(2);

console.log("Primitive Set contents:", primitiveSet);
console.log("Size of the set:", primitiveSet.size);

Primitive Set contents: Set(3) { 1, 2, 3 }
Size of the set: 3


### The Limitation of Native Sets

While the native `Set` correctly handles `string` and `number`, theoretical computer science frequently requires more complex structures like *sets that contain other sets* (e.g., powerset constructions or partitions). 

Let us observe what happens when we attempt to nest native `Set` objects. In the following example, we create three sets. We add the number `1` to two distinct sets, and then add both of these sets to a main set.

In [5]:
const mainSet = new Set<Set<number>>();
mainSet

Set(0) {}


In [6]:
const childSetA = new Set<number>();
childSetA.add(1);
childSetA

Set(1) { 1 }


In [7]:
const childSetB = new Set<number>();
childSetB.add(1);
childSetB

Set(1) { 1 }


In [8]:
// Adding structurally identical sets to the main set
mainSet.add(childSetA);
mainSet.add(childSetB);

console.log("Main Set contents:", mainSet);
console.log("Size of the main set:", mainSet.size);

Main Set contents: Set(2) { Set(1) { 1 }, Set(1) { 1 } }
Size of the main set: 2


### Analysis of the Output
The output demonstrates a critical failure in modeling mathematical sets. The `mainSet` contains two identical child sets (`Set(1) { 1 }`), resulting in a size of `2`. 

Mathematically, $\{ \{1\}, \{1\} \}$ must simplify to $\{ \{1\} \}$.

This occurs because the native `Set` in JavaScript uses **Reference Equality** to compare objects. It checks whether `childSetA` and `childSetB` occupy the exact same address in the computer's memory. Since they were instantiated separately via `new Set()`, they reside at different memory addresses, causing the computer to treat them as distinct elements despite their identical content.

In [9]:
childSetA == childSetB

false


## Introducing Value Semantics (Deep Equality)

We require a data structure that guarantees **Value Semantics** (also known as Deep Equality). Value semantics dictate that two structures are considered equal if their mathematical contents are identical, regardless of their memory location.

For this purpose, we utilize the `RecursiveSet` library.

In [10]:
import { RecursiveSet } from "recursive-set";

In [11]:
const recursiveMainSet = new RecursiveSet<RecursiveSet<number>>();

const recursiveChildA = new RecursiveSet<number>(1);
const recursiveChildB = new RecursiveSet<number>(1);

recursiveMainSet.add(recursiveChildA);
recursiveMainSet.add(recursiveChildB);

console.log("RecursiveSet contents:", recursiveMainSet);
console.log("Size of the RecursiveSet:", recursiveMainSet.size);

RecursiveSet contents: {{1}}
Size of the RecursiveSet: 1


## Representing Mathematical Tuples

In addition to sets, theoretical models frequently rely on **Tuples**. A tuple is a finite, ordered sequence of elements. Unlike sets, the order of elements in a tuple is strictly preserved, and duplicate elements are allowed.

The most common example of a tuple, which you already know from high school mathematics, is a **Coordinate Pair** $(x, y)$ in a 2D Cartesian coordinate system. 

For two points $P$ and $Q$:
$$ P = (2, 5) $$
$$ Q = (5, 2) $$

Because the order matters, $P \neq Q$. The $x$-coordinate and $y$-coordinate have strictly defined positions. 

Let us explore how to represent such coordinate tuples in TypeScript, and why we might want to store them in a Set (e.g., a set of visited coordinates on a grid).

## Using Arrays to Represent Tuples

The most intuitive approach is to use standard TypeScript arrays to represent these coordinate pairs, since arrays inherently preserve order. 

Let us define a set of coordinates that a robot has already visited, and then attempt to check if a specific coordinate is in that set.

In [12]:
const visitedCoordinates = new Set<number[]>();

const startPoint = [2, 5];
visitedCoordinates.add(startPoint);
visitedCoordinates

Set(1) { [ 2, 5 ] }


In [13]:
// The robot wants to check if it has already visited the point (2, 5)
const currentPosition = [2, 5];
const hasVisited = visitedCoordinates.has(currentPosition);

console.log("Robot has visited:", visitedCoordinates);
console.log("Has the robot visited (2, 5) before? ->", hasVisited);

Robot has visited: Set(1) { [ 2, 5 ] }
Has the robot visited (2, 5) before? -> false


### Analysis of the Output
Once again, the native data structures fail us. The `visitedCoordinates.has(currentPosition)` evaluates to `false`, even though the point `[2, 5]` was clearly added to the set.

As discussed in the previous section, this is due to **Reference Equality**. The computer treats `startPoint` and `currentPosition` as two completely distinct memory addresses, even though they represent the exact same mathematical point.

In [14]:
startPoint == currentPosition

false


Furthermore, native arrays introduce a second, highly dangerous flaw for mathematical modeling: **Mutability**. 
In mathematics, the point $(2, 5)$ is a constant, abstract concept. The number $2$ cannot suddenly become $99$. However, in TypeScript, arrays can be altered after their creation.

In [15]:
// Arrays are mutable (changeable)
startPoint[0] = 99;
console.log("Mutated Point P:", startPoint);

Mutated Point P: [ 99, 5 ]


In [16]:
visitedCoordinates

Set(1) { [ 99, 5 ] }


If we use arrays to represent coordinates, an accidental modification elsewhere in the code could silently corrupt our entire logic. We need a structure that provides **Deep Equality** (so it works inside Sets/Maps) and **Immutability** (so it cannot be altered).

## The `Tuple` Class

The `recursive-set` library provides a specialized `Tuple` class. A `Tuple` behaves exactly like a mathematical coordinate pair or sequence:
1. It is compared structurally (Deep Equality).
2. It is completely immutable; once created, it cannot be changed.

In [17]:
import { Tuple } from "recursive-set";

// Create a mathematically sound set of visited coordinates
const safeVisitedCoords = new RecursiveSet<Tuple<[number, number]>>();

const p1 = new Tuple<[number, number]>(2, 5);
safeVisitedCoords.add(p1);
safeVisitedCoords

{(2, 5)}


In [18]:
// We search using a completely new instance with the same mathematical content
const searchPoint = new Tuple<[number, number]>(2, 5);
const isPointFound = safeVisitedCoords.has(searchPoint);

console.log("Robot has visited:", safeVisitedCoords);
console.log("Has the robot visited (2, 5) before? ->", isPointFound);

Robot has visited: {(2, 5)}
Has the robot visited (2, 5) before? -> true


In [19]:
p1[0] = 3; 

3


In [20]:
safeVisitedCoords

{(2, 5)}


## Associating Data: The `RecursiveMap`

Now that we know how to define strictly correct coordinates using `Tuple`, we often want to associate data with these coordinates. In computer science, a data structure that associates a *Key* with a *Value* is called a **Map** (or Dictionary / Associative Array).

Think of a chess board or a grid: You have a coordinate (the Key) and a piece on that coordinate (the Value). Mathematically, this is a mapping: 
$$ f: (x, y) \rightarrow \text{Value} $$
$$ f((2, 5)) = \text{"Queen"} $$

Let us try to implement this mapping using the native `Map` in TypeScript.

In [21]:
const nativeGrid = new Map<number[], string>();

const posP = [2, 5];
nativeGrid.set(posP, "Queen");
nativeGrid

Map(1) { [ 2, 5 ] => 'Queen' }


In [22]:
// We try to look up what piece is at coordinate (2, 5)
const searchPos = [2, 5];
const pieceAtPos = nativeGrid.get(searchPos);

console.log("Piece at (2, 5) ->", pieceAtPos);

Piece at (2, 5) -> undefined


### Analysis of the Output
The output is `undefined`. Once again, the native data structure fails because the native `Map` uses **Reference Equality** for its keys. It searches for the exact memory address of `searchPos`, not the mathematical sequence `[2, 5]`.

In [23]:
searchPos == posP

false


The next test is even more enlightening.

In [24]:
[2, 5] == [2, 5]

1:1 - This condition will always return 'false' since JavaScript compares objects by reference, not value.


## The `RecursiveMap` Class

Just like `RecursiveSet`, the `recursive-set` library provides a class `RecursiveMap`. It ensures that **Value Semantics (Deep Equality)** are applied to the *Keys* of the map. 

Let us use our mathematically sound `Tuple` as the key in a `RecursiveMap`.

In [25]:
import { RecursiveMap } from "recursive-set";

// We define a map where the key is a Coordinate Tuple, and the value is a String
const safeGrid = new RecursiveMap<Tuple<[number, number]>, string>();

const safePos = new Tuple<[number, number]>(2, 5);
safeGrid.set(safePos, "Queen");
safeGrid

RecursiveMap(1) {
  (2, 5) => Queen
}


In [26]:
// Searching with a new, structurally identical tuple
const lookupTuple = new Tuple<[number, number]>(2, 5);
const foundPiece  = safeGrid.get(lookupTuple);

console.log("Safe Grid contents:\n", safeGrid);
console.log("Piece at (2, 5) ->", foundPiece);

Safe Grid contents:
 RecursiveMap(1) {
  (2, 5) => Queen
}
Piece at (2, 5) -> Queen


The `RecursiveMap` correctly returns `"Queen"`. Because the map relies on deep equality, it understands that the tuple `(2, 5)` used for storing is mathematically identical to the tuple `(2, 5)` used for looking up the value.

**Summary:**
1. **Uniqueness:** Use `RecursiveSet` when you need a mathematical set without duplicates.
2. **Sequences:** Use `Tuple` when you need an ordered, immutable pair or sequence (e.g., coordinates, transition pairs).
3. **Mappings:** Use `RecursiveMap` when you need to assign a value to a complex key (e.g., assigning a target state to a transition tuple).

# Advanced Operations with `RecursiveSet`

Now that we have established how to construct mathematically sound sets and tuples, we can perform actual computations with them. In theoretical computer science, we rarely just store elements; we manipulate them using set theory and functional programming.

## Basic Mutations: Add and Remove

Adding and removing elements from a `RecursiveSet` works exactly as expected.

In [27]:
const alphabet = new RecursiveSet<string>("a", "b", "c");

console.log("Initial alphabet:", alphabet);

// Removing an element
alphabet.remove("b");
console.log("After removing 'b':", alphabet);

// Adding an element
alphabet.add("d");
console.log("After adding 'd':", alphabet);

Initial alphabet: {a, b, c}
After removing 'b': {a, c}
After adding 'd': {a, c, d}


## Immutability (The Freeze Contract)

In mathematics, a set like $A = \{1, 2\}$ is a constant concept. If you use set $A$ as an element inside another set $B$, $A$ **must never change** again. If you were allowed to add a $3$ to $A$ later, it would silently corrupt the structure of $B$.

To prevent this dangerous behavior, `RecursiveSet` implements a strict **Freeze Contract**. The moment you insert a set (or a tuple) into another collection (like a `RecursiveSet` or `RecursiveMap`), it becomes **frozen** (immutable).

In [28]:
const stateA   = new RecursiveSet<number>(1, 2);
const powerset = new RecursiveSet<RecursiveSet<number>>();

// By adding stateA to the powerset, stateA is permanently FREEZES!
powerset.add(stateA);
powerset

{{1, 2}}


In [29]:
// Attempting to mutate a frozen set will throw an Error
stateA.add(3); 

/Users/stroetmann/Library/Mobile Documents/com~apple~CloudDocs/Box/Kurse/Logic/TypeScript/node_modules/recursive-set/dist/cjs/index.js:575
            throw new Error("Frozen Set modified.");
            ^

Error: Frozen Set modified.
    at RecursiveSet.add (/Users/stroetmann/Library/Mobile Documents/com~apple~CloudDocs/Box/Kurse/Logic/TypeScript/node_modules/recursive-set/dist/cjs/index.js:575:19)
    at evalmachine.<anonymous>:5:16
    at evalmachine.<anonymous>:7:3
    at sigintHandlersWrap (node:vm:280:12)
    at Script.runInThisContext (node:vm:135:14)
    at Object.runInThisContext (node:vm:317:38)
    at Object.execute (/Users/stroetmann/opt/anaconda3/envs/logic-ts/lib/node_modules/tslab/dist/executor.js:160:38)
    at JupyterHandlerImpl.handleExecuteImpl (/Users/stroetmann/opt/anaconda3/envs/logic-ts/lib/node_modules/tslab/dist/jupyter.js:250:38)
    at /Users/stroetmann/opt/anaconda3/envs/logic-ts/lib/node_modules/tslab/dist/jupyter.js:208:57
    at async JupyterHandlerImpl.h

What if we actually *need* to modify a set that has been frozen? We cannot change the original, because it is now securely locked inside the `powerset`. However, we can create a shallow, mutable copy of it using the `clone()` method.

In [30]:
const stateAModified = stateA.clone();
stateAModified.add(3);

console.log("Original stateA (still frozen inside powerset):", stateA.toString());
console.log("New stateAModified (unfrozen and independent):", stateAModified.toString());

Original stateA (still frozen inside powerset): {1, 2}
New stateAModified (unfrozen and independent): {1, 2, 3}


In [31]:
powerset

{{1, 2}}


## Mathematical Set Operations

The library provides all standard mathematical set operations built-in. These operations always return a **new** `RecursiveSet`, leaving the original sets unmodified.

- **Union** ($A \cup B$): Elements in $A$ or $B$.
- **Intersection** ($A \cap B$): Elements in both $A$ and $B$.
- **Difference** ($A \setminus B$): Elements in $A$ but not in $B$.
- **Symmetric Difference** ($A \bigtriangleup B$): Elements in either $A$ or $B$, but not in both.

In [32]:
const setX = new RecursiveSet<number>(1, 2, 3);
const setY = new RecursiveSet<number>(3, 4, 5);

const unionSet = setX.union(setY);
const intersectionSet = setX.intersection(setY);
const differenceSet = setX.difference(setY); // X without Y
const symDiffSet = setX.symmetricDifference(setY); // (X U Y) \ (X ∩ Y)

console.log(`X = ${setX}, Y = ${setY}`);
console.log("X ∪ Y (Union):", unionSet);
console.log("X ∩ Y (Intersection):", intersectionSet);
console.log("X \\ Y (Difference):", differenceSet);
console.log("X △ Y (Symmetric Difference):", symDiffSet);

X = {1, 2, 3}, Y = {3, 4, 5}
X ∪ Y (Union): {1, 2, 3, 4, 5}
X ∩ Y (Intersection): {3}
X \ Y (Difference): {1, 2}
X △ Y (Symmetric Difference): {1, 2, 4, 5}


### Subset and Superset Relations ($\subseteq$, $\supseteq$)
Two further fundamental relations in set theory are:
- $A \subseteq B$ (*A is a subset of B*): every element of $A$ is also contained in $B$.
- $A \supseteq B$ (*A is a superset of B*): $A$ contains all elements of $B$.

Note that $A = B$ implies both $A \subseteq B$ and $A \supseteq B$ simultaneously.

In [33]:
const small = new RecursiveSet<number>(2, 3);
const large = new RecursiveSet<number>(1, 2, 3, 4, 5);

console.log(`${small} ⊆ ${large}: ${small.isSubset(large)}`);
console.log(`${large} ⊇ ${small}: ${large.isSuperset(small)}`);
console.log(`${large} ⊆ ${small}: ${large.isSubset(small)}`);

{2, 3} ⊆ {1, 2, 3, 4, 5}: true
{1, 2, 3, 4, 5} ⊇ {2, 3}: true
{1, 2, 3, 4, 5} ⊆ {2, 3}: false


### Selecting a Random Element
In mathematical proofs, we frequently state: *"Let $x$ be a random element from the set $A$."* 
If you attempt to simulate this in code by iterating over the set and stopping after the first element, you will always get the exact same element. This is because `RecursiveSet` internally preserves the insertion order. To genuinely select an arbitrary element, you must use the `pickRandom()` method.

In [34]:
setX

{1, 2, 3}


In [35]:
console.log(`X = ${setX}`)
for (let i = 0; i < 5; ++i) {
    console.log("Random element x ∈ X:", setX.pickRandom());
}

X = {1, 2, 3}
Random element x ∈ X: 3
Random element x ∈ X: 3
Random element x ∈ X: 3
Random element x ∈ X: 1
Random element x ∈ X: 2


## Cartesian Product and Powerset

In addition to basic operations, set theory defines operations that generate entirely new, structurally distinct sets. You may recall these concepts from high school combinatorics or probability theory.

### Cartesian Product ($A \times B$)
The Cartesian product combines two sets into a set of all possible ordered pairs (Tuples), where the first element belongs to $A$ and the second belongs to $B$. 

A classic everyday example is finding all possible combinations of two independent choices, such as pairing a main dish with a drink from a menu.

In [37]:
const mainDishes = new RecursiveSet<string>("Burger", "Pasta");
const drinks     = new RecursiveSet<string>("Water", "Cola");

const mealCombinations = mainDishes.cartesianProduct(drinks);

console.log("Cartesian Product (A x B):");
mealCombinations;

Cartesian Product (A x B):
{(Burger, Cola), (Burger, Water), (Pasta, Cola), (Pasta, Water)}


### Powerset ($\mathcal{P}(A)$ or $2^A$)
The powerset is the set of all possible subsets of $A$, including the empty set ($\emptyset$) and the set $A$ itself. The size of the powerset is always $2^{|A|}$.

A common real-world analogy is choosing optional toppings for a pizza. Given a set of available toppings, you can choose none (the empty set), exactly one, any combination, or all of them. The powerset represents every possible customized pizza you could order.

In [38]:
const optionalToppings = new RecursiveSet<string>("Cheese", "Sauce", "Olives");
const allPossibleOrders = optionalToppings.powerset();

console.log("\nPowerset P(A):");
console.log(allPossibleOrders);
console.log("Size of the Powerset:", allPossibleOrders.size); // Expected: 2^3 = 8


Powerset P(A):
{{Cheese, Olives, Sauce}, {Cheese, Olives}, {Cheese, Sauce}, {Cheese}, {Olives, Sauce}, {Olives}, {Sauce}, {}}
Size of the Powerset: 8


*Warning:* The powerset grows exponentially. A set with just 20 elements will generate a powerset with $1,048,576$ subsets. The `RecursiveSet` library will intentionally throw an error if you attempt to compute a powerset for $|A| > 20$ to prevent your computer from running out of memory.

## Mathematical View of Functional Operations

Before using functional operations in TypeScript, it is helpful to describe them mathematically.

Let $A$ be a set, let $f : A \to B$ be a function, and let $P(x)$ be a predicate.
A predicate is a statement about an element $x$ that is either true or false.

In TypeScript, an expression such as `word => word.length === 3`
defines a predicate $P(word)$.
An expression such as `word => word.toUpperCase()`
defines a function $f(word)$.

### Set-builder notation
In mathematics, we often describe sets by specifying a property that their elements must satisfy:

$$
\{x \in A \mid P(x)\}
$$

This is read as:
“the set of all elements $x$ in $A$ such that $P(x)$ holds.”

### `filter(predicate)`
The operation `filter` keeps exactly those elements that satisfy the predicate:

$$
A\mathrm{.filter}(P) = \{x \in A \mid P(x)\}
$$

In [39]:
const vocabulary = new RecursiveSet<string>("cat", "dog", "bat", "mouse", "owl");

const threeLetterWords = vocabulary.filter(word => word.length == 3);

console.log("Original Vocabulary:", vocabulary);
console.log("Filtered (length == 3):", threeLetterWords);

Original Vocabulary: {bat, cat, dog, mouse, owl}
Filtered (length == 3): {bat, cat, dog, owl}


### `map(transform)`
The operation `map` applies a function to every element of a set:

$$
A\mathrm{.map}(f) = \{f(x) \mid x \in A\}
$$

Note that the result is again a set, so duplicate results are automatically removed.

In [40]:
"abc".toUpperCase()

ABC


In [41]:
// We transform every word to uppercase
const uppercaseWords = threeLetterWords.map(word => word.toUpperCase());
console.log("Uppercase Words:", uppercaseWords);

Uppercase Words: {BAT, CAT, DOG, OWL}


In [42]:
// Example of duplicate reduction during map:
// We map every word to its length. ("cat" -> 3, "bat" -> 3)
const wordLengths = threeLetterWords.map(word => word.length);
console.log("Word Lengths Set:", wordLengths); // Expected: {3}

Word Lengths Set: {3}


### `every(predicate)` and `some(predicate)`
These two operations correspond directly to logical quantifiers:

$$
A\mathrm{.every}(P) \iff \forall x \in A: P(x)
$$

$$
A\mathrm{.some}(P) \iff \exists x \in A: P(x)
$$

In [43]:
const numbers = new RecursiveSet<number>(2, 4, 6, 8);

const areAllEven = numbers.every(n => n % 2 == 0);
const containsEight = numbers.some(n => n == 8);
const containsOdd = numbers.some(n => n % 2 != 0);

console.log("Are all numbers even? ->", areAllEven);
console.log("Does the set contain an 8? ->", containsEight);
console.log("Does the set contain an odd number? ->", containsOdd);

Are all numbers even? -> true
Does the set contain an 8? -> true
Does the set contain an odd number? -> false


### `filterMap(predicate, transform)`
This operation combines filtering and mapping in a single step:

$$
A\mathrm{.filterMap}(P, f) = \{f(x) \mid x \in A \land P(x)\}
$$

In [44]:
const mixedNumbers = new RecursiveSet<number>(1, 2, 3, 4, 5);

// 1. filterMap: Keep only odd numbers AND multiply them by 10
const oddTens = mixedNumbers.filterMap(
    n => n % 2 == 1,    // Predicate (filter)
    n => n * 10         // Transform (map)
);
console.log("Odd numbers multiplied by 10:", oddTens);

Odd numbers multiplied by 10: {10, 30, 50}


### `flatMap(iterable, mapper)`
If each element $x$ is mapped to an entire set $f(x)$, then `flatMap` collects all of these sets and merges them into one:

$$
A\mathrm{.flatMap}(f) = \bigcup_{x \in A} f(x)
$$

In [45]:
// We start with a set containing only the number 0
const integerSet = new RecursiveSet<number>(0);

// We can use a simple native array as input to generate new elements
const naturalNumbers = [1, 2, 3];

// For every natural number n, we generate a set containing n and -n.
integerSet.flatMap(
    naturalNumbers, 
    n => new RecursiveSet<number>(n, -n)
);

console.log("\nGenerated Integer Set (in-place mutation):");
integerSet;


Generated Integer Set (in-place mutation):
{-3, -2, -1, 0, 1, 2, 3}



### `reduce(accumulator)`
The operation `reduce` combines all elements into a single value.
Mathematically, this is similar to operations such as summation or product:

$$
\sum_{x \in A} x
\qquad\text{or}\qquad
\prod_{x \in A} x
$$

More generally, `reduce` repeatedly combines an accumulator with the next element.

In [46]:
const costs = new RecursiveSet<number>(10, 20, 30);

// We start with an initial value of 0, and add each element to the accumulator
const totalSum = costs.reduce((acc, current) => acc + current, 0);

console.log("Set of costs:", costs);
console.log("Total sum of costs:", totalSum);

Set of costs: {10, 20, 30}
Total sum of costs: 60
